# Inspecting Feature Distributions Before Modeling

## Before you train any machine learning model, you must understand what your data looks like.

Not what you assume it looks like.  
Not what the dataset description claims.  
**What it actually looks like when inspected column by column.**

## Why This Matters

**Feature distribution inspection** is the process of examining how values in each feature are distributed:
- Range and extremes
- Central tendency (mean, median)
- Spread (standard deviation)
- Skewness and symmetry
- Frequency patterns
- Anomalies and outliers

Models learn patterns from data. **If your features contain extreme skew, outliers, incorrect encodings, or data entry errors, the model learns those patterns too.**

### You Cannot Fix What You Do Not Inspect

This step is often rushed or skipped by beginners eager to train models quickly. That is a mistake.

**By the end of this lesson, you should confidently answer:**

✓ Are numerical features well-behaved or highly skewed?  
✓ Do categorical features have rare or unexpected levels?  
✓ Are there extreme outliers that will distort learning?  
✓ Do features require transformation before modeling?  
✓ Are there hidden data quality issues?

---

# Why Feature Distribution Inspection Matters

Machine learning algorithms make assumptions about data — some explicit, some implicit.

| Algorithm | Assumptions | Sensitive To |
|-----------|-----------|---|
| **Linear Models** | Linearity, normality | Outliers, skewness |
| **Distance-Based** (KNN, SVM) | Feature scale | Outlier values, scale |
| **Tree-Based** | None (robust) | Extreme imbalance |
| **Neural Networks** | Normalized inputs | Input scale, distribution |

## If You Don't Inspect, You Risk:

❌ Feeding highly skewed variables into linear models  
❌ Allowing extreme outliers to dominate loss functions  
❌ Treating ordinal categories as nominal  
❌ Ignoring rare classes that distort evaluation  
❌ Scaling variables that shouldn't be scaled  
❌ Missing data quality issues entirely  

## Inspection Reduces Uncertainty

It allows preprocessing decisions to be **informed** rather than **reactive**.

The best ML practitioners understand their data deeply before modeling.

## Part 1: Inspecting Numerical Features

### Summary Statistics (First Level)

The first question when you open a dataset: **What do the numbers look like?**

```
df.describe() gives you:
- count: Non-null observations
- mean: Central tendency
- std: Spread around mean
- min/max: Range
- 25%, 50%, 75%: Quartiles (identifies skewness visually)
```

### What to Look For

| Statistic | Concern | Action |
|-----------|---------|--------|
| **Large std vs mean** | High variability/spread | Consider scaling, log transform |
| **min >> 0, max >>> 0** | Heavy right skew | Log or Box-Cox transform |
| **25%, 50%, 75% close together** | Left-skewed or capped | Check for artificial boundaries |
| **Missing count != total** | NaN values present | Decide: impute, remove, or flag |
| **median != mean** | Asymmetric distribution | Likely outliers or skewness |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Set styling for better visualizations
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 5)

# Example dataset: Customer churn with numerical features
np.random.seed(42)
n_samples = 1000

# Create realistic financial/customer data
data = {
    'customer_age': np.random.normal(45, 15, n_samples).clip(18, 80),
    'account_balance': np.random.exponential(5000, n_samples),  # Right-skewed
    'monthly_transactions': np.random.poisson(8, n_samples),    # Count data
    'days_active': np.random.gamma(shape=2, scale=1000, size=n_samples),
    'support_calls': np.random.poisson(2, n_samples),
    'target': np.random.binomial(1, 0.3, n_samples)  # Binary target (churn: 1/0)
}
df = pd.DataFrame(data)

# FIRST LOOK: Summary statistics
print("=" * 80)
print("SUMMARY STATISTICS - First Level Inspection")
print("=" * 80)
print(df.describe())
print("\nData types:")
print(df.dtypes)

### Visualizing distributions: Histograms

The **histogram** reveals the shape of distributions:

- **Bell curve (normal)**: Symmetric, centered, natural
- **Right-skewed (long tail right)**: Most values low, few extreme high values → log transform
- **Left-skewed**: Opposite of above → 1/x transforms, reflect + log
- **Bimodal/Multimodal**: Two+ peaks → separate by category or investigate subgroups
- **Uniform**: Flat distribution → often artificial, check boundaries
- **Heavy outliers**: Extreme points separate from main cluster → investigate or cap

In [ ]:
# HISTOGRAMS: Shape reveals necessary transformations
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

numerical_cols = ['customer_age', 'account_balance', 'monthly_transactions', 
                  'days_active', 'support_calls']

for idx, col in enumerate(numerical_cols):
    axes[idx].hist(df[col], bins=30, edgecolor='black', alpha=0.7, color='steelblue')
    axes[idx].set_title(f'{col}', fontsize=11, fontweight='bold')
    axes[idx].set_ylabel('Frequency')
    
    # Add median line for reference
    median_val = df[col].median()
    axes[idx].axvline(median_val, color='red', linestyle='--', linewidth=2, label=f'Median: {median_val:.1f}')
    axes[idx].legend()

axes[-1].axis('off')  # Hide last subplot
plt.suptitle('Numerical Feature Distributions - Notice the Shapes!', fontsize=13, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()

print("OBSERVATIONS FROM HISTOGRAMS:")
print(f"✓ customer_age: Bell curve - NORMAL")
print(f"✓ account_balance: Heavy RIGHT-SKEW - needs LOG transform")
print(f"✓ monthly_transactions: COUNT data (Poisson) - discrete")
print(f"✓ days_active: GAMMA distribution - right-skewed")
print(f"✓ support_calls: COUNT data - sparse")

### Detecting Outliers: Boxplots

The **boxplot** visualizes quartiles and outliers:

```
     IQR = Q3 - Q1
     Outlier threshold = Q3 + 1.5 * IQR  (or Q1 - 1.5 * IQR for low end)
     Points beyond threshold = outliers (plotted as dots)
```

**Outlier Decision Tree:**

```
Is this an outlier?
├─ YES, it's a measurement error → Remove
├─ YES, valid but extreme rare event → Cap at threshold
├─ YES, meaningful signal → Keep separate or transform
└─ NO, within reasonable range → Proceed
```

In [ ]:
# BOXPLOTS: Detect outliers and skewness at a glance
fig, axes = plt.subplots(1, 5, figsize=(16, 4))

for idx, col in enumerate(numerical_cols):
    # Create boxplot
    bp = axes[idx].boxplot(df[col], vert=True, patch_artist=True)
    bp['boxes'][0].set_facecolor('lightblue')
    
    # Add swarmplot for small datasets (shows actual points)
    if len(df) < 500:
        axes[idx].scatter(np.random.normal(1, 0.04, size=len(df)), df[col], 
                         alpha=0.3, s=20, color='steelblue')
    
    axes[idx].set_ylabel('Value')
    axes[idx].set_title(col, fontweight='bold')
    axes[idx].grid(axis='y', alpha=0.3)

plt.suptitle('Boxplots - Spotting Outliers and Skewness', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# QUANTITATIVE OUTLIER DETECTION
print("\n" + "="*80)
print("OUTLIER DETECTION: IQR Method")
print("="*80)

for col in numerical_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
    outlier_pct = len(outliers) / len(df) * 100
    
    print(f"\n{col}:")
    print(f"  Q1={Q1:.2f}, Q3={Q3:.2f}, IQR={IQR:.2f}")
    print(f"  Bounds: [{lower_bound:.2f}, {upper_bound:.2f}]")
    print(f"  Outlier Count: {len(outliers)} ({outlier_pct:.1f}%)")

### Quantifying Shape: Skewness & Kurtosis

**Skewness** measures asymmetry:
- **Skewness = 0**: Symmetric (normal)
- **Skewness > 0.5**: Right-skewed (tail goes right) → Consider log, sqrt, Box-Cox
- **Skewness < -0.5**: Left-skewed → 1/x transforms or reflection
- **|Skewness| < 0.5**: Approximately normal

**Kurtosis** measures tail heaviness:
- **Kurtosis ≈ 3 (excess=0)**: Normal
- **Kurtosis > 3**: Heavy tails (fat tails) → Outlier concerns
- **Kurtosis < 3**: Light tails → Data more concentrated

In [ ]:
# CALCULATE SKEWNESS AND KURTOSIS
print("\n" + "="*80)
print("SKEWNESS & KURTOSIS: Quantifying Distribution Shape")
print("="*80)

from scipy.stats import skew, kurtosis

summary_stats = []
for col in numerical_cols:
    skew_val = skew(df[col])
    kurt_val = kurtosis(df[col])
    
    # Classify skewness
    if abs(skew_val) < 0.5:
        skew_class = "Approximately Normal"
    elif skew_val > 0.5:
        skew_class = "RIGHT-SKEWED (needs transform)"
    else:
        skew_class = "LEFT-SKEWED (needs transform)"
    
    # Classify kurtosis
    excess_kurt = kurt_val - 3
    if excess_kurt > 1:
        kurt_class = "Heavy-tailed (outlier prone)"
    elif excess_kurt < -1:
        kurt_class = "Light-tailed"
    else:
        kurt_class = "Normal-like"
    
    summary_stats.append({
        'Feature': col,
        'Skewness': f"{skew_val:.3f}",
        'Class': skew_class,
        'Kurtosis': f"{kurt_val:.3f}",
        'Tail': kurt_class
    })
    
    print(f"\n{col}:")
    print(f"  Skewness: {skew_val:.3f} ({skew_class})")
    print(f"  Kurtosis: {kurt_val:.3f} ({kurt_class})")

# Show as table
summary_df = pd.DataFrame(summary_stats)
print("\n" + "="*80)
print("SUMMARY TABLE")
print("="*80)
print(summary_df.to_string(index=False))

---

## Part 2: Inspecting Categorical Features

### Value Counts & Class Balance

For categorical variables, the key question: **How distributed are the categories?**

**Ideal Scenario:**
- Classes relatively balanced (e.g., 40/60 split is acceptable)

**Warning Signs:**
- **Rare classes** (< 5% of data) → May need separate handling
- **Severe imbalance** (e.g., 1/99 split) → Class weights, resampling, stratified split needed
- **Dozens of categories** → Dimensionality explosion → Group rare categories together

In [ ]:
# ADD CATEGORICAL FEATURES to our example dataset
df['region'] = np.random.choice(['North', 'South', 'East', 'West'], size=len(df), p=[0.5, 0.3, 0.15, 0.05])
df['customer_type'] = np.random.choice(['Premium', 'Standard', 'Basic'], size=len(df), p=[0.1, 0.4, 0.5])
df['status'] = np.random.choice(['Active', 'Inactive', 'Dormant'], size=len(df))

categorical_cols = ['region', 'customer_type', 'status']

print("\n" + "="*80)
print("CATEGORICAL FEATURE INSPECTION: Value Counts")
print("="*80)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for idx, col in enumerate(categorical_cols):
    # Get value counts
    value_counts = df[col].value_counts()
    pct = (value_counts / len(df) * 100).round(1)
    
    # Barplot
    axes[idx].bar(range(len(value_counts)), value_counts.values, color='steelblue', edgecolor='black', alpha=0.7)
    axes[idx].set_xticks(range(len(value_counts)))
    axes[idx].set_xticklabels(value_counts.index, rotation=45, ha='right')
    axes[idx].set_ylabel('Count')
    axes[idx].set_title(col, fontweight='bold')
    axes[idx].grid(axis='y', alpha=0.3)
    
    # Add percentage labels on bars
    for i, (v, p) in enumerate(zip(value_counts.values, pct.values)):
        axes[idx].text(i, v + 5, f'{p}%', ha='center', fontsize=9, fontweight='bold')

plt.suptitle('Categorical Feature Distributions', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# TABLE: Detailed counts and percentages
print("\nDetailed Category Breakdown:")
for col in categorical_cols:
    print(f"\n{col}:")
    value_counts = df[col].value_counts()
    pct = (value_counts / len(df) * 100).round(2)
    
    for category, count in value_counts.items():
        p = pct[category]
        imbalance_flag = " ⚠️ RARE" if p < 5 else ""
        print(f"  {category}: {count} ({p}%){imbalance_flag}")

---

## Part 3: Target Distribution & Class Balance

### The Target Is Your North Star

Before training ANY model, always ask:

1. **How balanced is the target?** (crucial for classification)
2. **Does the minority class matter?** (sometimes imbalance is fine, sometimes critical)
3. **Which metric is important?** (precision vs recall vs F1 vs ROC-AUC)

**Common Scenarios:**

| Scenario | Target Split | Metric | Strategy |
|----------|-------------|--------|----------|
| **Balanced** | 45/55 | F1, Accuracy | Standard training fine |
| **Mildly Imbalanced** | 30/70 | F1, weighted metrics | Stratified split, class weights |
| **Severely Imbalanced** | 5/95 | Precision/Recall/ROC-AUC | SMOTE, undersampling, threshold tuning |
| **Extreme** | 1/99 | No accuracy! | Anomaly detection approach |

In [ ]:
# ANALYZE TARGET DISTRIBUTION
print("\n" + "="*80)
print("TARGET DISTRIBUTION: Understanding Your Prediction Task")
print("="*80)

target_counts = df['target'].value_counts().sort_index()
target_pct = (target_counts / len(df) * 100).round(2)

print(f"\nTarget Distribution:")
for target_val, count in target_counts.items():
    pct = target_pct[target_val]
    label = "Churn" if target_val == 1 else "Retain"
    print(f"  {label} ({target_val}): {count} samples ({pct}%)")

# Visualize target distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
axes[0].bar(['Retain (0)', 'Churn (1)'], target_counts.values, color=['green', 'red'], alpha=0.7, edgecolor='black')
axes[0].set_ylabel('Count')
axes[0].set_title('Target Distribution (Count)', fontweight='bold')
axes[0].grid(axis='y', alpha=0.3)

for i, (v, p) in enumerate(zip(target_counts.values, target_pct.values)):
    axes[0].text(i, v + 5, f'{p}%', ha='center', fontweight='bold')

# Pie chart
colors = ['green', 'red']
explode = (0.05, 0.05)
axes[1].pie(target_counts.values, labels=['Retain (0)', 'Churn (1)'], autopct='%1.1f%%', 
            colors=colors, explode=explode, startangle=90, textprops={'fontsize': 11, 'fontweight': 'bold'})
axes[1].set_title('Target Distribution (Proportion)', fontweight='bold')

plt.suptitle('Class Balance Assessment', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# FEATURE DISTRIBUTIONS BY TARGET
print("\n" + "="*80)
print("FEATURE DISTRIBUTIONS BY TARGET: Do Features Separate Classes?")
print("="*80)

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()

# Plot numerical features by target
for idx, col in enumerate(numerical_cols):
    for target_val in [0, 1]:
        data = df[df['target'] == target_val][col]
        axes[idx].hist(data, bins=20, alpha=0.6, label=f'Target={target_val}', edgecolor='black')
    
    axes[idx].set_title(col, fontweight='bold')
    axes[idx].set_ylabel('Frequency')
    axes[idx].legend()
    axes[idx].grid(axis='y', alpha=0.3)

plt.suptitle('Numerical Features: Distribution by Target Class', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n✓ ANALYSIS: Do distributions differ by target class?")
print("  → If YES: Feature likely PREDICTIVE")
print("  → If NO: Feature may be WEAK or REDUNDANT")

---

## Part 4: Data Transformations

### Transforming Skewed Features

**Why transform?**
- Linear models assume normality → Transform helps
- Distance-based models (SVM, KNN) benefit from normalized scale
- Tree-based models don't need transformation (they're scale-invariant)

**Common Transformations:**

| Transform | Use Case | Formula | Side Effects |
|-----------|----------|---------|--------------|
| **Log** | Right-skewed (most common) | $y' = \log(y)$ | Can't handle zeros, interpret as percentage |
| **Sqrt** | Moderate right-skew | $y' = \sqrt{y}$ | Also can't handle zeros |
| **Box-Cox** | Optimal power transform | Automatic optimization | Requires $y > 0$ |
| **Yeo-Johnson** | Works with negatives | Power transform variant | Better for mixed signs |
| **StandardScaler** | Normalize to mean=0, std=1 | $y' = \frac{y - \mu}{\sigma}$ | Doesn't fix skewness, just scale |
| **MinMaxScaler** | Scale to [0, 1] | $y' = \frac{y - \min}{\max - \min}$ | Sensitive to outliers |

In [ ]:
# TRANSFORMATIONS: Before vs After
from scipy.stats import boxcox, yeojohnson
from sklearn.preprocessing import StandardScaler, MinMaxScaler

# Focus on highly skewed feature: account_balance
feature_to_transform = 'account_balance'
original = df[feature_to_transform].copy()

# Apply transformations
df[f'{feature_to_transform}_log'] = np.log1p(original)  # log1p = log(1 + x) handles zeros
df[f'{feature_to_transform}_sqrt'] = np.sqrt(original)
df[f'{feature_to_transform}_boxcox'], lambda_param = boxcox(original + 0.1)  # Add small value to avoid zeros
df[f'{feature_to_transform}_standard'] = StandardScaler().fit_transform(original.values.reshape(-1, 1))

# Calculate skewness for comparison
print("="*80)
print(f"TRANSFORMATION IMPACT ON SKEWNESS: {feature_to_transform}")
print("="*80)

transforms = {
    'Original': original,
    'Log Transform': df[f'{feature_to_transform}_log'],
    'Sqrt Transform': df[f'{feature_to_transform}_sqrt'],
    'Box-Cox': df[f'{feature_to_transform}_boxcox'],
    'StandardScaler': df[f'{feature_to_transform}_standard']
}

for name, data in transforms.items():
    skew_val = skew(data)
    print(f"{name:20s}: Skewness = {skew_val:7.3f}")

# VISUALIZE TRANSFORMATIONS
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()

for idx, (name, data) in enumerate(transforms.items()):
    axes[idx].hist(data, bins=30, edgecolor='black', alpha=0.7, color='steelblue')
    axes[idx].set_title(f'{name}\n(Skewness: {skew(data):.3f})', fontweight='bold')
    axes[idx].set_ylabel('Frequency')
    axes[idx].grid(axis='y', alpha=0.3)

axes[-1].axis('off')
plt.suptitle(f'Comparing Transformation Methods: {feature_to_transform}', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n✓ INSIGHT: Which transformation should you use?")
print("  → Box-Cox: Best for statistical optimality (when all values > 0)")
print("  → Log: Simplest, most interpretable (especially for financial/count data)")
print("  → StandardScaler: Only fixes scale, NOT skewness")
print("  → No transform: OK for tree-based models")

---

## Part 5: Comprehensive Inspection Checklist

### Data Quality First

```
BEFORE TOUCHING ANY ALGORITHM:
─────────────────────────────────

NUMERICAL FEATURES:
☐ Review summary statistics (describe())
☐ Check for missing values (NaN count)
☐ Identify outliers (IQR method or visualize boxplots)
☐ Calculate skewness and kurtosis
☐ Visualize histograms (check for normality, modes, boundaries)
☐ Decide: transform? remove outliers? cap? impute?

CATEGORICAL FEATURES:
☐ Count unique values
☐ Check value_counts() distribution
☐ Identify rare categories (< 5%)
☐ Decide: group rare categories? one-hot encode?
☐ Detect ordinal vs nominal

TARGET:
☐ Check class balance (for classification)
☐ Identify minority/majority class
☐ Decide: class weights? stratified split? resampling?
☐ Understand metric requirements

FEATURE-TARGET RELATIONSHIP:
☐ Plot feature distributions by target class
☐ Do they differ? (Should mostly be YES for useful features)
☐ Check correlation (numerical) or chi-square (categorical)
☐ Identify suspected weak features

DATA INTEGRITY:
☐ No leakage (future data in features)
☐ No data quality warnings
☐ Date/time features handled properly
☐ Categorical encoding appropriate for model type
```

### Quick Decision Tree

```
For Each Feature:

1. Is it numerical or categorical?
   ├─ NUMERICAL: Go to 2
   └─ CATEGORICAL: Go to 4

2. Is it normally distributed? (Check histogram/skewness)
   ├─ YES: No transform needed → Go to 3
   ├─ NO (right-skew): Apply LOG → Go to 3
   └─ NO (left-skew, outliers): Apply transformation or remove outliers

3. Is it scaled appropriately?
   ├─ Tree-based model: No scaling needed → KEEP AS IS
   ├─ Linear/distance-based: Apply StandardScaler → READY
   └─ Done: Feature ready for modeling

4. (CATEGORICAL) How many unique values?
   ├─ 2: One-vs-All or direct encoding
   ├─ 3-10: One-hot encode (or ordinal if ordinal)
   └─ 10+: Group rare categories first, then encode

5. (CATEGORICAL) Are there missing values?
   ├─ YES: Treat as separate category or impute → Proceed
   └─ NO: Proceed to encoding
```

In [ ]:
# COMPLETE INSPECTION WORKFLOW: One feature end-to-end
def inspect_feature_complete(df, feature_name, target_name=None):
    """
    Comprehensive single-feature inspection pipeline.
    Shows all key statistics and visualizations.
    """
    print(f"\n{'='*80}")
    print(f"COMPLETE INSPECTION: {feature_name}")
    print(f"{'='*80}\n")
    
    data = df[feature_name].dropna()
    
    # 1. TYPE CHECK
    if pd.api.types.is_numeric_dtype(data):
        feature_type = "NUMERICAL"
    else:
        feature_type = "CATEGORICAL"
    
    print(f"Feature Type: {feature_type}")
    print(f"Non-null Count: {len(data)} / {len(df)} ({len(data)/len(df)*100:.1f}%)")
    print(f"Missing Values: {df[feature_name].isna().sum()}")
    
    if feature_type == "NUMERICAL":
        print(f"\n--- NUMERICAL FEATURE INSPECTION ---")
        print(f"Mean: {data.mean():.3f}")
        print(f"Std Dev: {data.std():.3f}")
        print(f"Min: {data.min():.3f}")
        print(f"Max: {data.max():.3f}")
        print(f"Range: {data.max() - data.min():.3f}")
        
        skew_val = skew(data)
        kurt_val = kurtosis(data)
        print(f"Skewness: {skew_val:.3f}", end="")
        if abs(skew_val) < 0.5:
            print(" → Approximately Normal")
        elif skew_val > 0.5:
            print(" → RIGHT-SKEWED (consider log transform)")
        else:
            print(" → LEFT-SKEWED (consider transform)")
        
        print(f"Kurtosis: {kurt_val:.3f}", end="")
        if kurt_val > 3:
            print(" → Heavy-tailed (outlier risk)")
        else:
            print(" → Normal-like")
        
        # Outliers
        Q1, Q3 = data.quantile(0.25), data.quantile(0.75)
        IQR = Q3 - Q1
        outliers = data[(data < Q1 - 1.5*IQR) | (data > Q3 + 1.5*IQR)]
        print(f"\nOutliers (IQR method): {len(outliers)} ({len(outliers)/len(data)*100:.1f}%)")
        
    else:
        print(f"\n--- CATEGORICAL FEATURE INSPECTION ---")
        vc = data.value_counts()
        print(f"Unique Values: {len(vc)}")
        pct = (vc / len(data) * 100).round(1)
        print("\nCategory Distribution:")
        for cat, count in vc.head(10).items():
            p = pct[cat]
            marker = " ⚠️" if p < 5 else ""
            print(f"  {cat}: {count} ({p}%){marker}")
        
        if len(vc) > 10:
            print(f"  ... and {len(vc) - 10} more categories")

# Run complete inspection on a couple features
inspect_feature_complete(df, 'account_balance')
inspect_feature_complete(df, 'region')
inspect_feature_complete(df, 'target')

print("\n" + "="*80)
print("✓ INSPECTION COMPLETE")
print("="*80)
print("\nNEXT STEPS:")
print("1. Record findings in a document")
print("2. Make transformation decisions (log? cap? group?)")
print("3. Implement preprocessing based on decisions")
print("4. Trust your data before building models")

---

## Key Takeaways

### The Three Pillars of Data Understanding

**1. Know Your Features**
   - Numerical: Understand shape, scale, skewness, outliers
   - Categorical: Understand cardinality, balance, rare classes
   - All: Check for missing values, data quality issues

**2. Know Your Target**
   - Classification: Check class balance, define metrics accordingly
   - Regression: Check distribution, understand range/scale
   - All: Ensure target sufficiently predictable from features

**3. Know Your Relationships**
   - Do features differ by target value?
   - Are there interactions between features?
   - Is any feature redundant with others?

### Common Mistakes to Avoid

| ❌ Mistake | ✅ Correct Approach |
|-----------|-------------------|
| Train model without looking at data | Spend 20% time on exploration |
| Use raw skewed features with linear models | Transform or use robust model |
| Ignore severe class imbalance | Use class weights or stratified sampling |
| Scale features before removing outliers | Remove/cap outliers FIRST, then scale |
| Don't know your test set distribution | Separate test set early, analyze separately |
| Apply same preprocessing to train & test | Fit preprocessing on TRAIN ONLY |

### Remember

**"All complex problems are solved by understanding data first."**

The time spent understanding distributions prevents:
- Training a model on bad data
- Making incorrect assumptions
- Missing critical data quality issues
- Building models that don't generalize
- Wasting time on unnecessary transformations

**Next: Apply these inspection techniques to your own data and make informed preprocessing decisions.**